# 02 - Preprocessing and Encoding (Robust Version)
Ce notebook montre exactement comment les traces brutes deviennent des entrees modeles avec justification scientifique des choix de longueur et des baselines.

## Choix retenus
- Longueur principale: p95 calcule sur train
- Troncature robuste: dual-view pre et post
- Baselines: BoW frequence, TF-IDF, bigrams
- Labels: binaire normal=0, attack=1

In [37]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import pickle
import sys

import numpy as np
import pandas as pd
from scipy import sparse
import yaml

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.encoding as encoding_module
importlib.reload(encoding_module)

from src.data_loading import parse_syscall_file
from src.encoding import (
    PAD_TOKEN,
    build_baseline_feature_sets,
    build_vocab,
    encode_sequence,
    make_bow_matrix,
    make_ngram_count_matrix,
    fit_tfidf_transformer,
)
from src.preprocessing import (
    compute_length_statistics,
    filter_invalid_sequences,
    pad_or_truncate,
    sanitize_sequence,
    suggest_truncation_lengths,
)
from src.utils import METADATA_ROOT, PROCESSED_ROOT, ensure_dir

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'default.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

In [31]:
index_path = METADATA_ROOT / 'adfa_sample_index_protocol_a.csv'
if not index_path.exists():
    raise FileNotFoundError('Lance le notebook 01_dataset_audit.ipynb avant celui-ci')

df = pd.read_csv(index_path)
valid_df, invalid_df = filter_invalid_sequences(df)

print('Valid traces:', len(valid_df))
print('Invalid traces:', len(invalid_df))
display(valid_df[['split', 'label']].value_counts().rename('count').reset_index())

Valid traces: 5951
Invalid traces: 0


,split,label,count
0,val,normal,2828
1,train,normal,1256
2,test,normal,1121
3,train,attack,510
4,test,attack,165
5,val,attack,71


In [32]:
sequences = []
for p in valid_df['filepath']:
    seq = parse_syscall_file(Path(p), strict=True)
    sequences.append(sanitize_sequence(seq))

valid_df['raw_sequence'] = sequences
valid_df['seq_len'] = valid_df['raw_sequence'].map(len)

train_lengths = valid_df[valid_df['split'] == 'train']['seq_len'].tolist()
stats = compute_length_statistics(train_lengths)
suggested = suggest_truncation_lengths(train_lengths, candidates=[0.90, 0.95, 0.99])

fixed_max_len = int(cfg.get('preprocessing', {}).get('fixed_max_len', 512))
length_comparison_df = pd.DataFrame([
    {'strategy': 'p95_train', 'max_len': int(suggested['p95'])},
    {'strategy': 'p99_train', 'max_len': int(suggested['p99'])},
    {'strategy': 'fixed', 'max_len': fixed_max_len},
])
display(length_comparison_df)

print('Train length stats:', json.dumps(stats, indent=2))
print('Suggested max lengths:', suggested)

max_len = int(suggested['p95'])
max_len_choice_summary = {
    'selected_strategy': 'p95_train',
    'selected_max_len': max_len,
    'p95': int(suggested['p95']),
    'p99': int(suggested['p99']),
    'fixed_max_len': fixed_max_len,
}
print('Final max_len choice summary:', json.dumps(max_len_choice_summary, indent=2))

,strategy,max_len
0,p95_train,1231
1,p99_train,1760
2,fixed,512


Train length stats: {
  "count": 1766.0,
  "min": 75.0,
  "max": 2948.0,
  "mean": 347.3346545866365,
  "median": 191.0,
  "p90": 825.5,
  "p95": 1231.25,
  "p99": 1760.0999999999995
}
Suggested max lengths: {'p90': 825, 'p95': 1231, 'p99': 1760}
Final max_len choice summary: {
  "selected_strategy": "p95_train",
  "selected_max_len": 1231,
  "p95": 1231,
  "p99": 1760,
  "fixed_max_len": 512
}


In [33]:
train_sequences = valid_df[valid_df['split'] == 'train']['raw_sequence'].tolist()
vocab = build_vocab(train_sequences, min_freq=1, max_vocab_size=None)
pad_value = vocab[PAD_TOKEN]

valid_df['encoded'] = valid_df['raw_sequence'].map(lambda s: encode_sequence(s, vocab))
valid_df['encoded_post'] = valid_df['encoded'].map(lambda s: pad_or_truncate(s, max_len=max_len, pad_value=pad_value, truncation='post', padding='post'))
valid_df['encoded_pre'] = valid_df['encoded'].map(lambda s: pad_or_truncate(s, max_len=max_len, pad_value=pad_value, truncation='pre', padding='post'))

valid_df['len_before_trunc'] = valid_df['encoded'].map(len)
valid_df['len_after_post'] = valid_df['encoded_post'].map(lambda s: sum(1 for x in s if x != pad_value))
valid_df['len_after_pre'] = valid_df['encoded_pre'].map(lambda s: sum(1 for x in s if x != pad_value))
valid_df['was_truncated'] = valid_df['len_before_trunc'] > max_len
valid_df['padding_ratio_post'] = valid_df['encoded_post'].map(lambda s: s.count(pad_value) / len(s) if len(s) > 0 else 0.0)
valid_df['padding_ratio_pre'] = valid_df['encoded_pre'].map(lambda s: s.count(pad_value) / len(s) if len(s) > 0 else 0.0)

trunc_rate_by_split = (
    valid_df.groupby('split')['was_truncated']
    .mean()
    .mul(100.0)
    .rename('truncated_rate_pct')
    .reset_index()
)
padding_rate_by_split = (
    valid_df.groupby('split')[['padding_ratio_post', 'padding_ratio_pre']]
    .mean()
    .mul(100.0)
    .reset_index()
)
trunc_cmp_df = valid_df[['trace_id', 'split', 'len_before_trunc', 'len_after_post', 'len_after_pre']].head(20)

print('Vocab size:', len(vocab))
display(valid_df[['trace_id', 'split', 'label', 'seq_len', 'len_after_post', 'len_after_pre', 'was_truncated']].head())
print('Taux de sequences tronquees par split (%)')
display(trunc_rate_by_split)
print('Taux moyen de padding par split (%)')
display(padding_rate_by_split)
print('Comparaison explicite post vs pre (extrait)')
display(trunc_cmp_df)

Vocab size: 152


,trace_id,split,label,seq_len,len_after_post,len_after_pre,was_truncated
0,T_45f8f1d3b4d8,test,attack,239,239,239,False
1,T_987315f46250,test,attack,735,735,735,False
2,T_e21193d96f6f,test,attack,691,691,691,False
3,T_4b1cc2748d83,test,attack,618,618,618,False
4,T_eb61364c0da3,test,attack,458,458,458,False


Taux de sequences tronquees par split (%)


,split,truncated_rate_pct
0,test,15.707621
1,train,5.039638
2,val,8.416695


Taux moyen de padding par split (%)


,split,padding_ratio_post,padding_ratio_pre
0,test,59.708629,59.708629
1,train,73.089856,73.089856
2,val,66.445445,66.445445


Comparaison explicite post vs pre (extrait)


,trace_id,split,len_before_trunc,len_after_post,len_after_pre
0,T_45f8f1d3b4d8,test,239,239,239
1,T_987315f46250,test,735,735,735
2,T_e21193d96f6f,test,691,691,691
3,T_4b1cc2748d83,test,618,618,618
4,T_eb61364c0da3,test,458,458,458
5,T_1b1827366f4f,test,447,447,447
6,T_279618ad7142,train,279,279,279
7,T_a28513e02680,train,766,766,766
8,T_0364ffbc2571,train,569,569,569
9,T_c592bc02c861,train,1068,1068,1068


In [34]:
train_seq = valid_df[valid_df['split'] == 'train']['raw_sequence'].tolist()
val_seq = valid_df[valid_df['split'] == 'val']['raw_sequence'].tolist()
test_seq = valid_df[valid_df['split'] == 'test']['raw_sequence'].tolist()

features = build_baseline_feature_sets(train_seq, val_seq, test_seq)

feature_dims_df = pd.DataFrame([
    {'feature_set': 'BoW', 'train_shape': tuple(features['bow']['train'].shape), 'val_shape': tuple(features['bow']['val'].shape), 'test_shape': tuple(features['bow']['test'].shape)},
    {'feature_set': 'TF-IDF', 'train_shape': tuple(features['tfidf']['train'].shape), 'val_shape': tuple(features['tfidf']['val'].shape), 'test_shape': tuple(features['tfidf']['test'].shape)},
    {'feature_set': 'Bigram', 'train_shape': tuple(features['bigram']['train'].shape), 'val_shape': tuple(features['bigram']['val'].shape), 'test_shape': tuple(features['bigram']['test'].shape)},
])

# Verification 1: vocab BoW doit provenir uniquement du train.
train_token_set = set(tok for seq in train_seq for tok in seq)
bow_vocab_token_set = set(features['bow']['vocab'].keys())
bow_vocab_train_only_check = bow_vocab_token_set.issubset(train_token_set)

# Verification 2: TF-IDF fit sur train uniquement (flag + shape consistency).
tfidf_flag = features.get('tfidf', {}).get('fit_on', '') == 'train'
tfidf_shape_consistency = (
    features['tfidf']['train'].shape[1] == features['tfidf']['val'].shape[1] == features['tfidf']['test'].shape[1]
 )
tfidf_fit_only_on_train_check = tfidf_flag and tfidf_shape_consistency

# Verification 3: Bigram fit sur train uniquement (flag + vocab consistency).
bigram_flag = features.get('bigram', {}).get('fit_on', '') == 'train'
bigram_train_vocab_size = features['bigram']['train'].shape[1]
bigram_val_vocab_size = features['bigram']['val'].shape[1]
bigram_test_vocab_size = features['bigram']['test'].shape[1]
bigram_shape_consistency = (bigram_train_vocab_size == bigram_val_vocab_size == bigram_test_vocab_size)
bigram_fit_only_on_train_check = bigram_flag and bigram_shape_consistency

fit_scope_flag = features.get('meta', {}).get('fit_scope') == 'train_only'

fit_checks = pd.DataFrame([
    {'check': 'vocab_built_only_on_train', 'value': bool(bow_vocab_train_only_check)},
    {'check': 'tfidf_fit_only_on_train', 'value': bool(tfidf_fit_only_on_train_check)},
    {'check': 'bigram_fit_only_on_train', 'value': bool(bigram_fit_only_on_train_check)},
    {'check': 'fit_scope_flag', 'value': bool(fit_scope_flag)},
])

display(feature_dims_df)
display(fit_checks)
failed_checks = fit_checks.loc[~fit_checks['value'], 'check'].tolist()
if failed_checks:
    raise ValueError(f"Verification encodage train-only echouee pour: {failed_checks}")

,feature_set,train_shape,val_shape,test_shape
0,BoW,"(1766, 150)","(2899, 150)","(1286, 150)"
1,TF-IDF,"(1766, 150)","(2899, 150)","(1286, 150)"
2,Bigram,"(1766, 2538)","(2899, 2538)","(1286, 2538)"


,check,value
0,vocab_built_only_on_train,True
1,tfidf_fit_only_on_train,True
2,bigram_fit_only_on_train,True
3,fit_scope_flag,True


In [35]:
sample_row = valid_df.iloc[0]
print('Example trace_id:', sample_row['trace_id'])
print('Raw sequence first 20:', sample_row['raw_sequence'][:20])
print('Encoded post first 20:', sample_row['encoded_post'][:20])
print('Encoded pre first 20:', sample_row['encoded_pre'][:20])

Example trace_id: T_45f8f1d3b4d8
Raw sequence first 20: [168, 265, 168, 168, 168, 265, 168, 168, 168, 168, 168, 168, 265, 168, 168, 265, 168, 168, 168, 265]
Encoded post first 20: [3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4]
Encoded pre first 20: [3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4]


In [38]:
ensure_dir(PROCESSED_ROOT)
ensure_dir(METADATA_ROOT)

protocol_used = str(valid_df['protocol'].mode().iloc[0]) if 'protocol' in valid_df.columns else 'protocol_a_main'
truncation_choice = 'dual_view_post_pre'
version_tag = f"{protocol_used}_ml{max_len}_dualview_v1"
version_dir = ensure_dir(PROCESSED_ROOT / version_tag)

required_columns = [
    'trace_id', 'filepath', 'split', 'label', 'label_binary',
    'protocol', 'len_before_trunc', 'len_after_post', 'len_after_pre',
    'was_truncated', 'padding_ratio_post', 'padding_ratio_pre',
]
missing_cols = [c for c in required_columns if c not in valid_df.columns]
if missing_cols:
    raise ValueError(f'Colonnes manquantes dans l index final: {missing_cols}')

valid_df['protocol_used'] = protocol_used
valid_df['truncation_choice'] = truncation_choice
valid_df['max_len_selected'] = int(max_len)
valid_df['encoding_version'] = version_tag

processed_index_path = version_dir / 'adfa_processed_index.parquet'
seq_post_path = version_dir / 'adfa_sequences_post.npy'
seq_pre_path = version_dir / 'adfa_sequences_pre.npy'
vocab_path = version_dir / 'token_vocab.json'
baseline_pack_path = version_dir / 'baseline_features.pkl'
invalid_path = version_dir / 'invalid_traces.csv'
frozen_config_path = METADATA_ROOT / 'frozen_preprocessing_config.json'
feature_dims_path = version_dir / 'feature_dimensions.csv'

if valid_df.empty:
    raise ValueError('Refus de sauvegarder: valid_df est vide')
if features['bow']['train'].shape[0] == 0 or features['bow']['train'].shape[1] == 0:
    raise ValueError('Refus de sauvegarder: features BoW train vides')

index_to_save = valid_df.drop(columns=['raw_sequence', 'encoded'])
index_to_save.to_parquet(processed_index_path, index=False)
np.save(seq_post_path, np.array(valid_df['encoded_post'].tolist(), dtype=np.int32))
np.save(seq_pre_path, np.array(valid_df['encoded_pre'].tolist(), dtype=np.int32))
invalid_df.to_csv(invalid_path, index=False)

with vocab_path.open('w', encoding='utf-8') as f:
    json.dump(vocab, f, indent=2, ensure_ascii=True)
with baseline_pack_path.open('wb') as f:
    pickle.dump(features, f)

feature_dims_df.to_csv(feature_dims_path, index=False)

frozen_config = {
    'generated_at': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'protocol_used': protocol_used,
    'heldout_families': cfg.get('protocols', {}).get('heldout_families', {}).get('families', []),
    'max_len_choice_summary': max_len_choice_summary,
    'truncation_choice': truncation_choice,
    'vocab_size': int(len(vocab)),
    'fit_checks': fit_checks.to_dict(orient='records'),
    'feature_dims': feature_dims_df.to_dict(orient='records'),
    'trunc_rate_by_split': trunc_rate_by_split.to_dict(orient='records'),
    'padding_rate_by_split': padding_rate_by_split.to_dict(orient='records'),
    'encoding_version': version_tag,
}
with frozen_config_path.open('w', encoding='utf-8') as f:
    json.dump(frozen_config, f, indent=2, ensure_ascii=True)

print('Saved:', processed_index_path)
print('Saved:', seq_post_path)
print('Saved:', seq_pre_path)
print('Saved:', vocab_path)
print('Saved:', baseline_pack_path)
print('Saved:', invalid_path)
print('Saved:', feature_dims_path)
print('Saved:', frozen_config_path)

Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_v1\adfa_processed_index.parquet
Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_v1\adfa_sequences_post.npy
Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_v1\adfa_sequences_pre.npy
Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_v1\token_vocab.json
Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_v1\baseline_features.pkl
Saved: C:\Users\223114186\Desktop\System_Call_Based_Intrusion_Detection\syscall-anomaly-detection\data\processed\protocol_a_main_ml1231_dualview_